In [1]:
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np
import os

In [2]:
csv_files = [
    "../Results/guitar_feature_database.csv",
    "../Results/piano_feature_database.csv",
    "../Results/saxophone_feature_database.csv",
    "../Results/violin_feature_database.csv"
]

In [3]:
# Kết nối Database
conn = psycopg2.connect(
    dbname="music_retrieval", user="postgres", password="admin", host="localhost", port="5433"
)
register_vector(conn)
cursor = conn.cursor()

In [ ]:
import json

# 1. Load tất cả CSV và tính mean / std toàn cục cho z-score normalization
mfcc_cols = [f"mfcc_{i}" for i in range(1, 14)]
chroma_labels = ["C", "Csharp", "D", "Dsharp", "E", "F", "Fsharp", "G", "Gsharp", "A", "Asharp", "B"]
chroma_cols = [f"chroma_{label}" for label in chroma_labels]
feature_cols = (
    mfcc_cols + chroma_cols +
    ["spectral_centroid", "spectral_bandwidth", "spectral_rolloff",
     "zero_crossing_rate", "tempo"]
)

all_dfs = []
for file_path in csv_files:
    if not os.path.exists(file_path):
        print(f"Không tìm thấy file: {file_path}. Bỏ qua...")
        continue
    all_dfs.append(pd.read_csv(file_path))

big_df = pd.concat(all_dfs, ignore_index=True)

stats = {}
for col in feature_cols:
    mean = float(big_df[col].mean())
    std = float(big_df[col].std())
    if std == 0 or pd.isna(std):
        std = 1.0  
    stats[col] = {"mean": mean, "std": std}

# Lưu để main.py dùng cùng tham số chuẩn hóa
stats_path = "../Results/feature_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"Đã lưu {stats_path} với {len(stats)} đặc trưng.")

def z(value, col):
    s = stats[col]
    return (float(value) - s["mean"]) / s["std"]

# 2. Xoá dữ liệu cũ (vector thô không thể trộn với vector đã chuẩn hoá)
cursor.execute("TRUNCATE music_segments RESTART IDENTITY")
conn.commit()
print("Đã clear bảng music_segments.")

# 3. Insert lại với z-score normalization
total_inserted = 0
for df in all_dfs:
    count = 0
    for _, row in df.iterrows():
        file_name = row['file_name']
        path = row['path']
        instrument = row['instrument']
        segment_id = row['segment_id']

        # Layer 1: Physical / Rhythmic (2 chiều, đã chuẩn hoá)
        v_layer1 = [
            z(row['zero_crossing_rate'], 'zero_crossing_rate'),
            z(row['tempo'], 'tempo'),
        ]

        # Layer 2: Perceptual / Texture (3 chiều, đã chuẩn hoá)
        v_layer2 = [
            z(row['spectral_centroid'], 'spectral_centroid'),
            z(row['spectral_bandwidth'], 'spectral_bandwidth'),
            z(row['spectral_rolloff'], 'spectral_rolloff'),
        ]

        # Layer 3: Identity / Voiceprint (25 chiều = 13 MFCC + 12 Chroma, đã chuẩn hoá)
        v_layer3 = [z(row[c], c) for c in mfcc_cols + chroma_cols]

        cursor.execute(
            """
            INSERT INTO music_segments
            (file_name, path, instrument, segment_id, layer1_physical, layer2_perceptual, layer3_identity)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            """,
            (file_name, path, instrument, segment_id, v_layer1, v_layer2, v_layer3),
        )
        count += 1
    conn.commit()
    total_inserted += count
    print(f"-> Đã chèn {count} segments (z-score).")

print(f"\nTổng cộng: {total_inserted} segments đã được chuẩn hoá và nạp vào DB.")

Đã lưu ../Results/feature_stats.json với 30 đặc trưng.
Đã clear bảng music_segments.
-> Đã chèn 1716 segments (z-score).
-> Đã chèn 3091 segments (z-score).
-> Đã chèn 473 segments (z-score).
-> Đã chèn 990 segments (z-score).

Tổng cộng: 6270 segments đã được chuẩn hoá và nạp vào DB.


In [5]:
cursor.close()
conn.close()
print(f"=== Đã lưu xong {total_inserted} segments dưới dạng Subvector! ===")

=== Đã lưu xong 6270 segments dưới dạng Subvector! ===
